# Crip_paly_scene_app

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 必要に応じて追加のライブラリをインストール
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm opencv-python

# 必要なパッケージのインストール
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall # numpyのバージョンを修正し、強制的に再インストール
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
import os
import sys

# プロジェクトのパスを指定（例: /content/drive/MyDrive/Visuable_for_you_tabletennis）
PROJECT_PATH = '/content/drive/MyDrive/Visuable_for_you_tabletennis'

# プロジェクトパスをPythonのパスに追加
sys.path.insert(0, PROJECT_PATH)

# 作業ディレクトリを変更
os.chdir(PROJECT_PATH)
print(f"作業ディレクトリ: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML

# Task1
from src.detection.table_detector import TableDetector
from src.detection.yolopose_tracker import YOLOPose_Tracker
from src.detection.player_classifier import PlayerClassifier
from src.detection.data_classes import TableInfo, PersonTrack
from src.detection.tracking_exporter import TrackingExporter
from src.visualization.player_classifier_visualizer import PlayerClassifierVisualizer

# Task2
# プロジェクトのモジュールをインポート
from src.models.play_classifier import PlayClassifierLSTM
from src.dataset.dataset import PoseSequenceDataset


print("✓ モジュールのインポートが完了しました")

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"OpenCVバージョン: {cv2.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
TABLE_MODEL = 'models/proto_type02_table_detection_models/best.pt'
POSE_MODEL = 'models/yolo11l-pose.pt'
LSTM_MODEL_PATH = 'models/proto_type03_crip_app_models/play_classifier_lstm.pth'
CONFIG_PATH = 'models/proto_type03_crip_app_models/player_classifier_config.json'

# 処理パラメータ
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Task1: 動画からプレイヤーの骨格データの獲得
FPS = 30.0              # 処理FPS（GPU利用時は高FPS推奨）
MAX_PLAYERS = 4         # 最大プレイヤー数
MIN_PLAYER_SCORE = 0.3  # プレイヤー判定の最小スコア閾値（0.0-1.0）
MIN_CONSECUTIVE_FRAMES = 30  # CS最小連続フレーム数
MAX_FRAME_GAP = 5           # 連続性を判定する際の最大フレーム間隔

save_video = True  # 骨格データを描画した動画を保存するかどうか

print("処理パラメータ:")
print(f"  処理FPS: {FPS}")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE}")
print(f"  最小連続フレーム数: {MIN_CONSECUTIVE_FRAMES}")
print(f"  最大フレーム間隔: {MAX_FRAME_GAP}")
print(f"  保存: {save_video}")

In [ ]:
INPUT_VIDEO = 'data/sample_videos/sample_video_01.mp4'  # 入力動画ファイルパス
OUTPUT_VIDEO = 'aaa'
CSV_OUTPUT = 'aaa'

In [ ]:
# メイン処理
def run_player_classification():
    """プレイヤー分類テストを実行"""
    
    print(f"\n動画ファイルを開いています: {INPUT_VIDEO}...")
    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        print("エラー: 動画ファイルを開けませんでした")
        return
    print("✓ 動画ファイルを開きました")
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\n入力情報:")
    print(f"  解像度: {width}x{height}")
    print(f"  動画FPS: {video_fps:.2f}")
    print(f"  総フレーム数: {total_frames}")
    print(f"  処理FPS: {FPS:.2f}")
    print(f"  出力動画FPS: {FPS:.2f} (処理したフレームのみ出力)")
    print(f"  保存モード: {'ファイルに保存' if save_video else 'メモリに保持'}\n")

    # フレーム間隔を計算（四捨五入で正確に）
    frame_interval = max(1, round(video_fps / FPS))
    
    print("コンポーネントを初期化しています...")
    table_detector = TableDetector(yolo_model_path=TABLE_MODEL)
    pose_tracker = YOLOPose_Tracker(model_path=POSE_MODEL, device = 'cuda')
    player_classifier = PlayerClassifier(max_players=MAX_PLAYERS,min_player_score=MIN_PLAYER_SCORE)
    
    # 常に初期化（メモリ保持のため）
    visualizer = PlayerClassifierVisualizer(table_detector, pose_tracker, player_classifier)
    csv_exporter = TrackingExporter()
    
    # 動画保存は save_video=True の時のみ
    video_writer = None
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, FPS, (width, height))

    # 卓球台を検出
    print("卓球台を検出中...")
    table_info = None
    max_detection_attempts = 100
    for attempt in range(max_detection_attempts):
        ret, frame = cap.read()
        if not ret:
            print("エラー: 動画の終端に達しました")
            cap.release()
            if video_writer:
                video_writer.release()
            return
        table_info = table_detector.detect_table_from_frame(frame, frame_idx=attempt, force_detect=True)
        if table_info is not None:
            print(f"✓ 卓球台を検出しました（フレーム {attempt + 1}、信頼度: {table_info.confidence:.2f}）\n")
            break
    if table_info is None:
        print(f"エラー: 卓球台を検出できませんでした")
        cap.release()
        if video_writer:
            video_writer.release()
        return

    # 動画を最初に戻す
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame_count = 0
    processed_count = 0
    player_ids = set()
    print("処理開始...\n")
    print(f"  フレーム間隔: {frame_interval} ({frame_interval}フレームごとに1回処理)")
    print(f"  予測処理フレーム数: 約{total_frames // frame_interval}フレーム")
    print(f"  処理率: {100.0 / frame_interval:.1f}%\n")

    # プログレスバー付きで処理
    pbar = tqdm(total=total_frames, desc="Processing")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            pbar.update(1)
            if frame_count % frame_interval != 0:
                continue
            processed_count += 1
            
            persons = pose_tracker.track_frame_with_table_filter(frame, table_info)
            if table_info and persons:
                player_classifier.update(persons, table_info, frame_count)

            if table_info:
                selected_ids, removed_ids = player_classifier.classify_players()
                player_ids = set(selected_ids)
                if removed_ids:
                    pose_tracker.remove_validated_track_ids(removed_ids)

            # プレイヤーの骨格データを常にメモリに保持
            if player_ids:
                player_persons = [p for p in persons if p.track_id in player_ids]
                if player_persons:
                    timestamp = frame_count / video_fps
                    csv_exporter.add_frame(frame_count, timestamp, player_persons)

            # 結果を描画（常に実行してメモリに保持）
            display_frame = visualizer.draw_results(frame, table_info, persons, player_ids)
            display_frame = visualizer.draw_candidate_info(display_frame, player_ids)

            # フレーム情報を表示
            cv2.putText(
                display_frame,
                f"Frame: {frame_count}/{total_frames} (Processed: {processed_count})",
                (10, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

            cv2.putText(
                display_frame,
                f"Detected: {len(persons)} persons, Players: {len(player_ids)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )
            
            # 動画保存は save_video=True の時のみ
            if video_writer:
                video_writer.write(display_frame)

    finally:
        pbar.close()
        cap.release()
        if video_writer:
            video_writer.release()

        print(f"\n✓ 処理完了:")
        print(f"  処理フレーム数: {frame_count}")
        print(f"  実際に処理したフレーム数: {processed_count}")
        print(f"  検出されたプレイヤーID: {sorted(player_ids)}")
        print(f"  候補者数: {len(player_classifier.candidates)}")

        if player_classifier.candidates:
            print(f"\n=== 候補者詳細 ===")
            candidates = []
            for track_id, candidate in player_classifier.candidates.items():
                if candidate.total_frames >= player_classifier.min_tracking_frames:
                    score = player_classifier._calculate_player_score(candidate)
                    candidates.append((track_id, candidate, score))
            candidates.sort(key=lambda x: x[2], reverse=True)

            for track_id, candidate, score in candidates:
                is_player = track_id in player_ids
                print(f"\nID {track_id} {'[PLAYER]' if is_player else ''}:")
                print(f"  スコア: {score:.3f}")
                print(f"  フレーム数: {candidate.total_frames}")
                print(f"  総運動量: {candidate.total_movement:.1f}")
                print(f"  卓球台付近比率: {candidate.near_table_ratio:.1%}")
        
        # 連続性フィルタリングを適用（常に実行）
        csv_exporter.filter_by_consecutive_frames(
            min_consecutive_frames=MIN_CONSECUTIVE_FRAMES,
            max_frame_gap=MAX_FRAME_GAP
        )

        # save_video フラグに基づいて保存または表示
        if save_video:
            # 動画とCSVをファイルに保存
            print(f"\n出力ビデオ: {OUTPUT_VIDEO} ({FPS:.1f}fps)")
            
            player_roles = {track_id: "player" for track_id in player_ids}
            csv_exporter.export_csv(CSV_OUTPUT, player_roles)
            print(f"プレイヤー骨格データをCSVに保存しました: {CSV_OUTPUT}")
        else:
            # メモリに保持（保存しない）
            print(f"\n出力ビデオ: メモリに保持（保存なし）")
            print(f"プレイヤー骨格データ: メモリに保持（保存なし）")
            print(f"  総フレーム数（フィルタ後）: {len(csv_exporter.frames)}")

# テストを実行
run_player_classification()

In [ ]:
# 設定ファイルを読み込み
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    print("設定ファイルを読み込みました:")
    for key, value in config.items():
        print(f"  {key}: {value}")
else:
    # デフォルト設定
    print("警告: 設定ファイルが見つかりません。デフォルト設定を使用します。")
    config = {
        'model_type': 'lstm',
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.3,
        'no_attention': False,
        'sequence_length': 30
    }

# モデル作成
print("\nモデルを作成中...")
model = PlayClassifierLSTM(
    input_size=34,
    hidden_size=config['hidden_size'],
    num_layers=config['num_layers'],
    dropout=config['dropout'],
    use_attention=not config.get('no_attention', False)
)

# 重み読み込み
checkpoint = torch.load(LSTM_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

print(f"モデル読み込み完了: {LSTM_MODEL_PATH}")
print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
print(f"Best Val F1: {checkpoint.get('best_val_f1', 'N/A')}")

In [ ]:
# データセット作成
sequence_length = config.get('sequence_length', 30)
dataset = PoseSequenceDataset(
    csv_path=POSE_CSV_PATH,
    label_path=None,  # ラベルなし（予測のみ）
    sequence_length=sequence_length,
    stride=1  # 全フレームを予測するためstride=1
)

print(f"予測開始:")
print(f"  入力CSV: {POSE_CSV_PATH}")
print(f"  総フレーム数: {len(dataset.data_df)}")
print(f"  シーケンス数: {len(dataset)}")
print(f"  シーケンス長: {sequence_length}フレーム")

# 各フレームの予測確率を集計
frame_probs = {}  # {frame_num: [probs]}

with torch.no_grad():
    for features, _, metadata in tqdm(dataset, desc="予測中"):
        # バッチ次元を追加
        features = features.unsqueeze(0).to(DEVICE)
        
        # 予測
        outputs = model(features)  # (1, seq, 1)
        probs = outputs.squeeze().cpu().numpy()  # (seq,)
        
        # フレームごとに確率を記録
        start_frame = metadata['start_frame']
        for i, prob in enumerate(probs):
            frame_num = start_frame + i
            if frame_num not in frame_probs:
                frame_probs[frame_num] = []
            frame_probs[frame_num].append(prob)

# 各フレームの確率を平均
predictions = []
for frame_num in sorted(frame_probs.keys()):
    avg_prob = np.mean(frame_probs[frame_num])
    prediction = 1 if avg_prob >= THRESHOLD else 0
    predictions.append({
        'frame': frame_num,
        'probability': avg_prob,
        'prediction': prediction,
        'num_predictions': len(frame_probs[frame_num])
    })

result_df = pd.DataFrame(predictions)

# 統計
num_play_frames = np.sum(result_df['prediction'] == 1)
play_ratio = num_play_frames / len(result_df) * 100
print(f"\n予測結果:")
print(f"  プレー中フレーム: {num_play_frames} / {len(result_df)} ({play_ratio:.1f}%)")

# CSV保存
output_csv_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
result_df.to_csv(output_csv_path, index=False)
print(f"  予測結果保存: {output_csv_path}")

# 先頭5件を表示
print("\n予測結果（先頭5件）:")
display(result_df.head())